In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os

import matplotlib as mpl
from matplotlib import pyplot as plt
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
from teeplot import teeplot as tp

import pylib


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-12-btr-foliage")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/vjhgs/download"),
)
df_pure


In [ ]:
df_sweep = pfl.alifestd_join_roots(
    pd.read_parquet("https://osf.io/download/ajdsz"),
)
df_sweep


In [ ]:
dfs = []
for df in (df_pure, df_sweep):
    df["x"] = df["position"] // df["nCol"]
    df["x_"] = df["x"] / df["nRow"]
    df["y"] = df["position"] % df["nCol"]
    df["y_"] = df["y"] / df["nCol"]

    df["origin_time"] = df["dstream_rank"]

    dfs.append(df)

df_pure, df_sweep = dfs


In [ ]:
pure_sample = pfl.alifestd_downsample_tips_asexual(
    df_pure, n_downsample=8_000
)
sweep_sample = pfl.alifestd_downsample_tips_asexual(
    df_sweep, n_downsample=8_000
)


## Plot Layer Tree


In [ ]:
for s, regime, layout in it.product(
    (1, 3),
    ("pure", "sweep"),
    ("vertical",),
):
    df = {
        "pure": pure_sample,
        "sweep": sweep_sample,
    }[regime]
    with tp.teed(
        pylib.chloropleth.draw_ctree,
        df,
        x="x_",
        y="y_",
        cmap=pylib.cmap.bcyr.get_color,
        layout=layout,
        scatter_kws=dict(
            edgecolor="none",
            s=s,
        ),
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                color="gainsboro",
                linewidth=0.5,
            ),
            margins=-0.05,
        ),
        teeplot_dpi=600,
        teeplot_outattrs={"regime": regime, "s": s},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.invert_yaxis()
        teed.figure.set_size_inches(3, 1.33)


## Plot Fossil Scatterplot


In [ ]:
for regime in "pure", "sweep":
    df = {
        "pure": df_pure,
        "sweep": df_sweep,
    }[regime]
    with tp.teed(
        pylib.chloropleth.draw_cscatter,
        df.pipe(
            pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
        ).dropna(subset=["x_", "y_"]),
        x="x",
        y="y",
        cmap=pylib.cmap.bcyr.get_color,
        despine=True,
        major=100,
        minor=25,
        xmax=1170,
        ymax=755,
        scatter_kws=dict(
            alpha=0.8,
            s=5,
        ),
        teeplot_dpi=600,
        teeplot_outattrs={"cmap": "bcyr", "regime": regime},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.set_aspect("equal")
        fig = teed.figure
        fig.set_size_inches(3, 2)
        fig.tight_layout()


## Combined Plots


In [ ]:
with tp.teed(
    plt.subplots,
    ncols=2,
    nrows=2,
    figsize=(6, 4),
    gridspec_kw={"height_ratios": [2, 1], "hspace": 0.1, "wspace": 0.05},
    teeplot_subdir=teeplot_subdir,
) as fig:
    fig, axs = fig

    for col, regime in enumerate(("pure", "sweep")):
        df = {
            "pure": df_pure,
            "sweep": df_sweep,
        }[regime]

        pylib.chloropleth.draw_cscatter(
            df.copy().pipe(
                pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
            ).dropna(subset=["x_", "y_"]),
            x="x",
            y="y",
            cmap=pylib.cmap.bcyr.get_color,
            despine=True,
            major=100,
            minor=25,
            scatter_kws=dict(
                alpha=0.8,
                s=3,
            ),
            xmax=1170,
            ymax=755,
            ax=axs[0, col],
        )
        axs[0, col].set_aspect("equal", anchor="S")

        pylib.chloropleth.draw_ctree(
            df.copy().pipe(
                pfl.alifestd_downsample_tips_asexual, n_downsample=5_000, seed=1
            ),
            x="x_",
            y="y_",
            cmap=pylib.cmap.bcyr.get_color,
            layout="vertical",
            scatter_kws=dict(
                edgecolor="none",
                s=1,
            ),
            scatter_shuffle=1,
            tree_kws=dict(
                edge=dict(
                    color="gainsboro",
                    linewidth=0.5,
                ),
                margins=-0.05,
            ),
            ax=axs[1, col],
        )
        if col == 1:
            axs[1, col].invert_xaxis()
        axs[1, col].invert_yaxis()
        axs[1, col].set_anchor("N")
